# 04 · Typed decision model as a LangGraph router node

An agent system usually starts with a **routing step**: read the request, decide which specialist handles it,
how urgent it is, and whether a human must step in. Using a big generative LLM for that step is slow and its output
has to be parsed. A **typed decision model** returns exactly the fields we need, in milliseconds.

Here we plug our `DecisionModel` into a [LangGraph](https://langchain-ai.github.io/langgraph/) workflow:

```
START → router ──(route / requires_human / low confidence)──► billing_agent
                                                            ├► orders_agent
                                                            ├► ... one node per team
                                                            └► human_handoff
        every branch → END
```

The specialist "agents" are stubs (they just write a reply); the point is the routing node.
By default the router is **XGBoost** (trained here in seconds, CPU is fine). Set `USE_LAYA = True` to swap in a Laya
checkpoint instead. Nothing else in the graph changes, because both share the `DecisionModel` interface.

## Setup

Works both on **Colab** and locally. On Colab this clones the repo and moves into it;
locally (opened from `notebooks/`) it just moves up to the repo root so `import src...` works.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/AdityaThakare72/typed-decision-models.git"
BRANCH = "main"

if not os.path.exists("src/dataset.py"):
    if os.path.exists("../src/dataset.py"):      # running locally from notebooks/
        os.chdir("..")
    else:                                         # fresh Colab runtime
        if not os.path.exists("typed-decision-models"):
            subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL], check=True)
        os.chdir("typed-decision-models")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

Install LangGraph (plus the usual light deps).

In [ ]:
%pip install -q langgraph scikit-learn xgboost pandas

## 1. Pick the router model

- `USE_LAYA = False` → train XGBoost on our train split (seconds).
- `USE_LAYA = True` → load a Laya checkpoint: your fine-tune from notebook 02 (`LAYA_PATH`), or a Hub ID for zero-shot.
  Needs `%pip install laya` and ideally a GPU runtime.

In [ ]:
from src.dataset import load_splits, ROUTES
from src.models.classical_ml import ClassicalDecisionModel

USE_LAYA = False
LAYA_PATH = "artifacts/laya-finetuned"   # or "convaiinnovations/laya" for zero-shot

train_df, val_df, test_df = load_splits("data")

if USE_LAYA:
    os.environ["USE_TF"] = "0"
    from src.models.laya_model import LayaDecisionModel
    router_model = LayaDecisionModel(LAYA_PATH, name="laya:router").load()
else:
    router_model = ClassicalDecisionModel("xgb").fit(train_df)
print("router model:", router_model.name)

## 2. Graph state

The **state** is the shared notebook every node reads from and writes to as the request flows through the graph.
A `TypedDict` is just a dict with declared keys, which documents what each node can expect.

`trace` uses a *reducer* (`operator.add`): when a node returns `{"trace": [...]}`, LangGraph **appends** to the list
instead of overwriting it, so we get a step-by-step log for free.

In [ ]:
import operator
from typing import Annotated, TypedDict


class TicketState(TypedDict, total=False):
    ticket: dict                              # the incoming request (query + metadata)
    decision: dict                            # filled in by the router
    reply: str                                # filled in by whichever agent handles it
    trace: Annotated[list[str], operator.add] # appended to by every node

## 3. The router node

It calls `predict_one` on the ticket and stores the typed decision in the state. One safety rule on top:
if the model's confidence is below `CONFIDENCE_FLOOR`, hand off to a human regardless of what it predicted.
(Confidence from Laya is uncalibrated out of the box, so tune this threshold on validation data.)

In [ ]:
CONFIDENCE_FLOOR = 0.5


def router_node(state: TicketState) -> dict:
    out = router_model.predict_one(state["ticket"])
    note = (f"router[{router_model.name}]: route={out.route} priority={out.priority} "
            f"human={out.requires_human} conf={out.confidence:.2f} ({out.latency_ms:.1f} ms)")
    return {"decision": out.to_dict(), "trace": [note]}


def pick_next_node(state: TicketState) -> str:
    """The conditional edge: read the typed decision and return the name of the next node."""
    decision = state["decision"]
    if decision["requires_human"] or decision["confidence"] < CONFIDENCE_FLOOR:
        return "human_handoff"
    return decision["route"] + "_agent"

## 4. Specialist nodes (stubs)

In a real system each of these would be its own LLM agent with tools (refund API, order DB, ...). Here they just write
a reply that shows the decision reached them, including the priority, which a real agent would use to pick an SLA.

In [ ]:
def make_agent(team: str):
    def agent_node(state: TicketState) -> dict:
        priority = state["decision"]["priority"]
        reply = f"[{team} agent | {priority} priority] handling: {state['ticket']['query']}"
        return {"reply": reply, "trace": [f"{team}_agent: replied"]}
    return agent_node


def human_handoff(state: TicketState) -> dict:
    d = state["decision"]
    reply = f"[HUMAN QUEUE | {d['priority']}] for {d['route']} team: {state['ticket']['query']}"
    return {"reply": reply, "trace": ["human_handoff: queued for a person"]}

## 5. Wire the graph

`add_conditional_edges` takes the router's output (`pick_next_node`) and a map from returned names to nodes.
Every specialist then goes to `END`.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TicketState)
builder.add_node("router", router_node)
builder.add_node("human_handoff", human_handoff)
for route in ROUTES:
    builder.add_node(route + "_agent", make_agent(route))

builder.add_edge(START, "router")
destinations = {route + "_agent": route + "_agent" for route in ROUTES}
destinations["human_handoff"] = "human_handoff"
builder.add_conditional_edges("router", pick_next_node, destinations)
for node in destinations:
    builder.add_edge(node, END)

graph = builder.compile()
# Mermaid text needs no extra packages; paste it into https://mermaid.live to see the picture.
print(graph.get_graph().draw_mermaid())

## 6. Try a few tickets

Hand-written tickets, including a calm billing question, an angry enterprise outage, and a fraud report.

In [ ]:
examples = [
    {"query": "can I get a refund for my last annual payment thanks", "account_tier": "free",
     "channel": "email", "urgency_score": 0.2, "past_failures": 0},
    {"query": "your site is down and I can't access reports we are losing money", "account_tier": "enterprise",
     "channel": "phone", "urgency_score": 0.95, "past_failures": 2},
    {"query": "someone logged into my account from Tokyo, I think I was hacked", "account_tier": "pro",
     "channel": "chat", "urgency_score": 0.7, "past_failures": 0},
    {"query": "where is my order 5531, it has not arrived", "account_tier": "pro",
     "channel": "chat", "urgency_score": 0.3, "past_failures": 0},
]

for ticket in examples:
    final = graph.invoke({"ticket": ticket})
    print("\n".join(final["trace"]))
    print("->", final["reply"], "\n")

## 7. How much of the graph's time is the router?

Run 100 test tickets through the whole graph and compare **router-node latency** with **end-to-end graph latency**
(p50 / p95). The gap is LangGraph's own overhead; with real LLM agents downstream, the router should be a rounding error.

In [ ]:
import time
import numpy as np
import pandas as pd

router_ms, total_ms, destinations_hit = [], [], []
for ticket in test_df.head(100).to_dict(orient="records"):
    start = time.perf_counter()
    final = graph.invoke({"ticket": ticket})
    total_ms.append((time.perf_counter() - start) * 1000)
    router_ms.append(float(final["trace"][0].rsplit("(", 1)[1].split(" ms")[0]))
    destinations_hit.append(final["trace"][-1].split(":")[0])

print(f"router node : p50 {np.percentile(router_ms, 50):.1f} ms | p95 {np.percentile(router_ms, 95):.1f} ms")
print(f"whole graph : p50 {np.percentile(total_ms, 50):.1f} ms | p95 {np.percentile(total_ms, 95):.1f} ms")
pd.Series(destinations_hit).value_counts()

## 8. Did tickets land in the right place?

Compare where each ticket *should* have gone (human if `requires_human`, else its route's agent) with where the graph
actually sent it. This is the number an ops team cares about.

In [ ]:
expected = ["human_handoff" if row["requires_human"] else row["route_target"] + "_agent"
            for row in test_df.head(100).to_dict(orient="records")]
correct = np.mean([e == d for e, d in zip(expected, destinations_hit)])
print(f"correct destination: {correct:.1%} of 100 tickets (confidence floor = {CONFIDENCE_FLOOR})")

**Things to try:**
- Flip `USE_LAYA = True` and rerun: same graph, different brain.
- Raise `CONFIDENCE_FLOOR` and watch more tickets go to humans (safer, more expensive).
- Replace a stub agent with a real LLM agent; the router stays the same.